# 중단된 영상 보완 학습 복구: 98–100 epoch

이 노트북은 run `20261120T123735Z_2e0735a1`의 **검증된 97 epoch 체크포인트**에서 남은 3 epoch만 이어 학습합니다. 총 목표는 `epochs=100`이며, 처음부터 100 epoch를 다시 학습하지 않습니다.

먼저 기존 Colab에서 **다시 연결**을 확인하세요. 기존 학습이 진행 중이면 이 복구 노트북을 실행하지 마세요. 중단된 상태라면 GPU 런타임을 선택하고 이 노트북에서 **런타임 → 모두 실행**, 필요한 경우 Drive 연결을 허용하세요. 같은 런타임의 학습 프로세스가 실행 중이거나 원본 백업이 97 epoch 이후로 바뀌었다면 자동으로 중지합니다. 이 검사는 다른 런타임에서 진행 중인 작업을 완전히 확인하는 수단은 아니므로 먼저 기존 상태를 확인해야 합니다.

원본 Drive 백업·기존 모델·이전 영상은 보존합니다. 새 실행 폴더에 CSV 1–97행과 이전 best.pt를 복원하고, optimizer·EMA·AMP scaler를 이어 사용합니다. 학습 시간은 중단 시간을 제외한 누적 실행 시간으로 기록하며, 난수 상태까지 동일한 무중단 실행을 보장하는 것은 아닙니다.

GPU 0, batch 16, imgsz 640, workers 4, seed 42와 기존 설정을 유지합니다. 이 복구 경로에서 오류가 나면 설정을 임의 변경하거나 100 epoch를 다시 시작하지 않고 로그를 보존합니다. 완료 후 원래 test 73장 평가, 추가 학습 장면 재검사, 결과 ZIP 저장을 실행합니다. 같은 영상의 결과는 독립 영상 성능이 아닌 **학습 장면 재검사**입니다.


## 1. 복구 실행 경로 설정
기존 실행을 다시 시작하는 셀이 아닙니다. 새 복구 ID와 새 저장 경로를 만들며 원본 백업은 읽기 전용으로 사용합니다. 같은 런타임에서 이 설정 셀을 반복 실행하지 마세요.


In [ ]:
from pathlib import Path
import datetime
import json
import os
import re
import shutil
import subprocess
import sys
import uuid

if not Path('/content').is_dir():
    raise RuntimeError('Run this notebook in Google Colab; local CPU fallback is disabled.')
if 'RECOVERY_RUN_ID' in globals():
    raise RuntimeError('Recovery already configured in this notebook; inspect its current state instead of rerunning all cells.')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RECOVERY_RUN_ID = RUN_ID
SOURCE_RUN_ID = '20261120T123735Z_2e0735a1'
REPOSITORY_URL = 'https://github.com/jeongjunseo/yolov8-stationery-tracking.git'
REPO_REF = 'main'
REPO = Path('/content') / ('stationery_video_finetune_' + RUN_ID)
DRIVE_PROJECT = Path('/content/drive/MyDrive/yolov8-stationery-tracking')
DRIVE_ADDITION = DRIVE_PROJECT / 'video_finetune/video_training_addition_v2.zip'
ADDITION_SHA256 = '22a7de84e451f3ad9c120773b74f42111c3955864ed21013d7614e83a980f4c9'
BASELINE_SHA256 = '72eec58f57a0decd694a98606e9759824cb7cb4d75af345e9ae8db735a4d66ea'
DATA_ARCHIVE = Path('/content/stationery_dataset.zip')
BACKUP_SOURCE = DRIVE_PROJECT / 'video_finetune/colab_runs' / SOURCE_RUN_ID
BACKUP_RUN = DRIVE_PROJECT / 'video_finetune/colab_runs' / RUN_ID
ADDITION_ROOT = REPO / 'runs/video_finetune_data/addition'
MIXED_DATA = REPO / 'runs/video_finetune_data/mixed.yaml'
print('Recovery run ID:', RUN_ID)
print('Source completed epochs: 97; remaining epochs: 98, 99, 100')

# Do not alter dependencies or start duplicate training in an active runtime.
import psutil
for proc in psutil.process_iter(['pid', 'cmdline']):
    try:
        args = proc.info['cmdline'] or []
        if proc.pid == os.getpid():
            continue
        names = [Path(arg).name for arg in args]
        active = ('finetune_video.py' in names and 'train' in args) or 'resume_video_finetune.py' in names
        if active:
            raise RuntimeError(f'Training is already active (PID {proc.pid}); preserve that process and inspect it first.')
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        continue


## 2. GPU와 Drive 연결 확인
CPU 학습은 허용하지 않습니다. GPU를 사용할 수 없으면 중단합니다. 필요한 Drive 연결 창을 완료하세요.


In [ ]:
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-c',
    "import torch; assert torch.cuda.is_available(), 'CUDA GPU is required'; "
    "print('PyTorch:', torch.__version__); print('GPU:', torch.cuda.get_device_name(0))"], check=True)
from google.colab import drive
drive.mount('/content/drive')
if not DRIVE_ADDITION.is_file():
    raise FileNotFoundError(f'The reviewed video-frame ZIP is missing: {DRIVE_ADDITION}')
if not re.fullmatch('[0-9a-f]{64}', ADDITION_SHA256):
    raise RuntimeError('Publish the reviewed addition ZIP SHA-256 before running this notebook.')
if not BACKUP_SOURCE.is_dir():
    raise FileNotFoundError(f'Original recovery backup missing: {BACKUP_SOURCE}')
BACKUP_RUN.mkdir(parents=True, exist_ok=False)
print('Every-epoch backups and final ZIP:', BACKUP_RUN)


## 3. 별도 checkout과 의존성 준비
새 checkout만 사용합니다. 공개된 과거 결과는 해시 기록과 함께 별도 스냅샷에 보존하며, 기존 학습 폴더를 초기화하지 않습니다. Colab의 CUDA PyTorch를 유지하고 검증된 Ultralytics 8.4.215를 설치합니다.


In [ ]:
if REPO.exists():
    raise FileExistsError(f'Preserving existing checkout: {REPO}')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPOSITORY_URL, str(REPO)], check=True)
# This runs only immediately after the guarded fresh clone, before sessions/data.
snapshot_result = subprocess.run(
    [sys.executable, str(REPO / 'scripts/preserve_published_finetune.py'), str(REPO)],
    check=True, capture_output=True, text=True, encoding='utf-8')
PUBLISHED_SNAPSHOT = json.loads(snapshot_result.stdout)
print('Previously published artifacts preserved:', PUBLISHED_SNAPSHOT['snapshot_directory'])
for required in ('scripts/finetune_video.py', 'scripts/resume_video_finetune.py', 'scripts/check_dataset.py', 'models/best.pt'):
    if not (REPO / required).is_file():
        raise FileNotFoundError(f'Publish the adaptation code before running: {required}')
SESSION = REPO / 'runs/video_finetune_sessions' / RUN_ID
SESSION.mkdir(parents=True, exist_ok=False)
OUT = REPO / 'outputs/video_finetune'
if OUT.exists():
    raise FileExistsError('This checkout already contains adaptation outputs; preserve that experiment.')
OUT.mkdir(parents=True)
(OUT / 'published_snapshot.json').write_text(json.dumps(PUBLISHED_SNAPSHOT, indent=2) + '\n', encoding='utf-8')
from importlib.metadata import version
constraints = SESSION / 'constraints.txt'
constraints.write_text('torch==' + version('torch') + '\n' + 'torchvision==' + version('torchvision') + '\n', encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt'),
                'ultralytics==8.4.215', '-c', str(constraints), '--disable-pip-version-check'], check=True)
pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], text=True, capture_output=True)
print(pip_check.stdout, pip_check.stderr)
(OUT / 'pip_check.json').write_text(json.dumps({'exit_code': pip_check.returncode,
    'stdout': pip_check.stdout, 'stderr': pip_check.stderr}, indent=2) + '\n', encoding='utf-8')
environment_code = """
import json, platform, subprocess
from pathlib import Path
from importlib.metadata import version
import torch, ultralytics, cv2
assert torch.cuda.is_available(), 'CUDA became unavailable after dependency installation'
result = {'runtime': 'Google Colab', 'python': platform.python_version(),
    'packages': {n: version(n) for n in ('torch','torchvision','ultralytics','opencv-python','PyYAML','lap')},
    'cuda_available': torch.cuda.is_available(), 'cuda_runtime': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0), 'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory,
    'device': '0', 'git_commit': subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()}
Path('outputs/video_finetune/environment.json').write_text(json.dumps(result, indent=2) + '\\n', encoding='utf-8')
print(json.dumps(result, indent=2))
"""
subprocess.run([sys.executable, '-c', environment_code], cwd=REPO, check=True)
for name in ('environment.json', 'pip_check.json', 'published_snapshot.json'):
    shutil.copy2(OUT / name, BACKUP_RUN / name)
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'scripts'))
from colab_support import digest_file
assert digest_file(REPO / 'models/best.pt') == BASELINE_SHA256
print('Verified baseline checkpoint; checkout:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())

SOURCE_ROOT = SESSION / 'recovery_source'
SOURCE_ROOT.mkdir(exist_ok=False)
SOURCE_FILES = {
    'last.pt': ('stationery_video_b16/weights/last.pt', 'c336147c23c3acef1fe55e39a603cb3afb79e8c82bb4f66bca4be0ef18bb8551'),
    'best.pt': ('stationery_video_b16/weights/best.pt', '61e35e27335a72841e5ded5075d1e05d76460a0aa4b03590c0c656b6f9121976'),
    'results.csv': ('stationery_video_b16/results.csv', '68055f924f75ca6008d417eb6ef98f150a86aa7ca6533a23c9dc3c0a7a47e77b'),
    'args.yaml': ('stationery_video_b16/args.yaml', '812cc28358fa2d993bad8bdfd2eb086f88c12e581fe6c69b448fa50ea4664841'),
    'backup_events_b16.json': ('backup_events_b16.json', 'b6f07b7f35a123cb84e1ea0266085bbf9b7032ef99dce7c4c9f07cf47ff79110'),
}
for name, (relative, expected) in SOURCE_FILES.items():
    source = BACKUP_SOURCE / relative
    if digest_file(source) != expected:
        raise RuntimeError(f'Original backup changed: {source}. Inspect its latest progress; do not restart it from epoch 97.')
    with source.open('rb') as src, (SOURCE_ROOT / name).open('xb') as dst:
        shutil.copyfileobj(src, dst)
    assert digest_file(SOURCE_ROOT / name) == expected
for name in ('environment.json', 'dataset_check.json', 'baseline_dataset_check.json'):
    with (BACKUP_SOURCE / name).open('rb') as src, (SOURCE_ROOT / name).open('xb') as dst:
        shutil.copyfileobj(src, dst)
# Recheck the source after copying, in case a still-running job advanced it.
for relative, expected in SOURCE_FILES.values():
    assert digest_file(BACKUP_SOURCE / relative) == expected, 'Source advanced during recovery staging; inspect the original runtime.'
print('Verified and preserved 97-epoch checkpoint, previous best, CSV, settings and backup history.')


## 4. 원래 데이터와 추가 프레임 재검증
보관된 원본 데이터 조각과 추가 프레임 ZIP을 해시로 검증합니다. train 546장(기존 498+추가 48), validation 116장, test 73장을 유지합니다. 원본 데이터의 분할이나 라벨을 다시 만들지 않습니다.


In [ ]:
from colab_support import prepare_dataset_archive, extract_dataset
transfer = prepare_dataset_archive(DATA_ARCHIVE, DRIVE_PROJECT)
extraction = extract_dataset(DATA_ARCHIVE, REPO)
subprocess.run([sys.executable, 'scripts/check_dataset.py'], cwd=REPO, check=True)
(OUT / 'baseline_dataset_check.json').write_text(json.dumps(
    {'transfer': transfer, 'extraction': extraction, 'check_dataset': 'passed'}, indent=2) + '\n', encoding='utf-8')
subprocess.run([sys.executable, 'scripts/finetune_video.py', 'prepare',
    '--archive', str(DRIVE_ADDITION), '--archive-sha256', ADDITION_SHA256,
    '--addition-root', str(ADDITION_ROOT), '--data', str(MIXED_DATA)], cwd=REPO, check=True)
for name in ('baseline_dataset_check.json', 'dataset_check.json'):
    shutil.copy2(OUT / name, BACKUP_RUN / name)
print('Mixed data:', MIXED_DATA.read_text())


## 5. 97 epoch 체크포인트에서 남은 3 epoch 재개
첫 학습 출력은 **98/100**이어야 합니다. 1/100을 표시하는 신규 학습 경로는 사용하지 않습니다. 97행의 CSV와 기존 best.pt를 보존하고, 새 백업 폴더에 98–100 epoch를 추가합니다. 원본 백업은 수정하지 않습니다.


In [ ]:
def run_logged(command, log_path):
    print('Running:', ' '.join(map(str, command)))
    with Path(log_path).open('x', encoding='utf-8') as log:
        process = subprocess.Popen(list(map(str, command)), cwd=REPO,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            encoding='utf-8', errors='replace', bufsize=1,
            env={**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONIOENCODING': 'utf-8'})
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        return process.wait()


log_path = SESSION / 'resume_epochs_98_100.log'
# Data restoration can take several minutes; recheck the original before resuming.
for relative, expected in SOURCE_FILES.values():
    assert digest_file(BACKUP_SOURCE / relative) == expected, 'Original backup advanced while restoring data; inspect its current state before recovery.'
subprocess.run([sys.executable, 'scripts/resume_video_finetune.py', 'prepare',
    '--source-root', str(SOURCE_ROOT), '--data', str(MIXED_DATA),
    '--backup-root', str(BACKUP_RUN)], cwd=REPO, check=True)
returncode = run_logged([sys.executable, '-u', 'scripts/resume_video_finetune.py', 'train',
    '--data', MIXED_DATA, '--backup-root', BACKUP_RUN], log_path)
shutil.copy2(log_path, BACKUP_RUN / log_path.name)
if returncode:
    raise RuntimeError(f'Recovery stopped; preserve and inspect {log_path}. No new 100-epoch run or automatic batch change was started.')
training = json.loads((OUT / 'training_run.json').read_text())
assert training['epochs_completed'] == 100 and training['device'].startswith('cuda')
assert training['actual_batch'] == 16
assert digest_file(REPO / 'models/best.pt') == BASELINE_SHA256
ACTUAL_BATCH = training['actual_batch']
print(json.dumps(training, indent=2))


## 6. Evaluate the unchanged original test split
These metrics are written separately from the baseline. The original dataset's known annotation and scene-overlap limitations still apply. The source-video frames are not in this test split.


In [ ]:
status = run_logged([sys.executable, '-u', 'scripts/finetune_video.py', 'evaluate',
    '--data', MIXED_DATA, '--batch', ACTUAL_BATCH, '--backup-root', BACKUP_RUN], SESSION / 'evaluate.log')
if status:
    raise RuntimeError('Original-test evaluation failed; inspect the saved log.')
print((OUT / 'test_comparison.json').read_text())


## 7. Recheck the added training scene with both models
Both models use imgsz 640 and the same labeled video frames. For the adapted model this is **training_scene_resubstitution**, not independent validation and not a new video benchmark. It is not used to select a checkpoint.


In [ ]:
status = run_logged([sys.executable, '-u', 'scripts/finetune_video.py', 'scene-recheck',
    '--batch', ACTUAL_BATCH, '--backup-root', BACKUP_RUN], SESSION / 'training_scene_recheck.log')
if status:
    raise RuntimeError('Training-scene recheck failed; inspect the saved log.')
for kind in ('baseline', 'finetuned'):
    print((OUT / f'training_scene_recheck_{kind}.json').read_text())


## 8. Collect only the new experiment artifacts
The ZIP contains the adapted model, new measured JSON, and new train/test graphs. It does not overwrite or republish baseline files. After downloading, run the separate adapted-model video, visually inspect actual classes/IDs/trails, and label that video as a training-scene demonstration in the report.


In [ ]:
archive = Path('/content') / f'stationery_video_finetune_results_{RUN_ID}.zip'
subprocess.run([sys.executable, 'scripts/finetune_video.py', 'collect', str(archive),
                '--backup-root', str(BACKUP_RUN)], cwd=REPO, check=True)
print('Final result ZIP:', archive)
print('Drive result folder:', BACKUP_RUN)
from IPython.display import FileLink, display
display(FileLink(str(archive)))

receipt = {
    'run_id': RUN_ID, 'archive_name': archive.name,
    'bytes': archive.stat().st_size, 'sha256': digest_file(archive),
    'drive_archive_sha256': digest_file(BACKUP_RUN / archive.name),
    'recorded_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
assert receipt['sha256'] == receipt['drive_archive_sha256'], 'Drive result ZIP differs from local Colab ZIP.'
with (BACKUP_RUN / 'collection_receipt.json').open('x', encoding='utf-8') as stream:
    json.dump(receipt, stream, indent=2)
print('Separate Drive checksum receipt:', json.dumps(receipt, indent=2))


## 오류 또는 연결 끊김이 다시 발생한 경우
원래 노트북과 이 복구 노트북의 **모두 실행을 반복하지 마세요**. 마지막 출력과 새 복구 run ID를 확인하세요. 새 Drive 폴더에는 진행한 epoch와 로그가 보존됩니다. 추가 복구는 그 최신 파일을 확인한 뒤 수행해야 합니다.

GPU 사용량 제한이나 할당 실패 시 CPU로 전환하지 않습니다. 새 GPU를 사용할 수 있을 때까지 체크포인트를 보존합니다. 사용자 요청으로 일시정지한 Windows CPU 학습과는 별개입니다.

참고: [프로젝트](https://github.com/jeongjunseo/yolov8-stationery-tracking), [Ultralytics 학습 문서](https://docs.ultralytics.com/modes/train/).
